In [1]:
# import sys
# !{sys.executable} -m pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

D:\Langchain\ModelComponent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\kuldi\AppData\Local\Temp\ipykernel_20716\2685459506.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - indexing

In [4]:
video_id = "pQa_tWZmlGs" # Here give only ID not full URL
try:
    # If you don’t care which language, this returns the “best” one
    ytt_api = YouTubeTranscriptApi()
    transcript = ytt_api.fetch(video_id)  # It will get all transcript of video

    # Flatten it to plain text
    transcript_text = " ".join(chunk.text for chunk in transcript)
    print(transcript_text)  # It will join all part of transcript.

except TranscriptsDisabled:
    print("No captions available for this video.")

Suppose you love math, and you had to choose just one proof to show someone to explain  why it is that math is beautiful, something that can be appreciated by anyone from a wide  range of backgrounds while still capturing the spirit of progress and cleverness in math. What would you choose? After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses,  published as a guest video over on MinutePhysics,  someone on Reddit asked about why the definition of an ellipse given in that video,  the classic two thumbtacks and a piece of string construction,  is the same as the definition involving slicing a cone. Well, my friend, you've asked about one of my all-time favorite proofs,  a lovely bit of 3D geometry which, despite requiring almost no background,  still captures the spirit of mathematical inventiveness. For context and to make sure we're all on the same page,  there are at least three main ways you could define an ellipse geometrically. One is to say you take

In [5]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Suppose you love math, and you had to choose just one proof to show someone to explain ', start=3.88, duration=4.585), FetchedTranscriptSnippet(text='why it is that math is beautiful, something that can be appreciated by anyone from a wide ', start=8.465, duration=4.744), FetchedTranscriptSnippet(text='range of backgrounds while still capturing the spirit of progress and cleverness in math.', start=13.209, duration=4.691), FetchedTranscriptSnippet(text='What would you choose?', start=18.3, duration=0.82), FetchedTranscriptSnippet(text="After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses, ", start=20.06, duration=4.424), FetchedTranscriptSnippet(text='published as a guest video over on MinutePhysics, ', start=24.484, duration=2.543), FetchedTranscriptSnippet(text='someone on Reddit asked about why the definition of an ellipse given in that video, ', start=27.027, duration=4.272), FetchedTranscri

## Step1b - Indexing (Text Splitting)

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript_text])

In [8]:
len(chunks)
chunks[0]

Document(metadata={}, page_content="Suppose you love math, and you had to choose just one proof to show someone to explain  why it is that math is beautiful, something that can be appreciated by anyone from a wide  range of backgrounds while still capturing the spirit of progress and cleverness in math. What would you choose? After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses,  published as a guest video over on MinutePhysics,  someone on Reddit asked about why the definition of an ellipse given")

## Step 1c & 1d -(Embedding Generation and storing in vector Store)

In [10]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(chunks, embeddings)

In [11]:
vector_store.index_to_docstore_id

{0: '852bc9f5-e31b-4e59-97f6-821a7030c98c',
 1: 'ca0f33f3-9c22-4b2f-a6d4-91883ee380ff',
 2: '55fb6342-56de-4489-8681-e6a2adbd0200',
 3: '70e0e488-f630-43f2-900e-4c0cb236289f',
 4: 'fb4057ff-9aff-405e-bfed-feb4b9b5ace0',
 5: '96da7906-5aba-4250-affd-0131871ef181',
 6: '28f85f91-5c85-4f31-bb9e-0f03fba0425a',
 7: '0cfe5b7d-415a-44a8-ac74-769d92d0da1b',
 8: '5e5d4ceb-8892-4e00-b1aa-35702266a305',
 9: '5f2ab01e-afdf-43c1-8ed3-8add71541df3',
 10: '3373db90-9d22-4583-9895-fa716a63dc2c',
 11: 'afbb1656-e97c-4602-a3e1-259630939b15',
 12: '5e91dbce-dba8-4882-a6fd-63d4d028c21f',
 13: '1707c6a1-00f4-4b1b-aa34-f9e23f74c412',
 14: 'ebf2b020-9f23-4330-9d78-b5f54c178ff7',
 15: 'fb0168c4-4ae2-4c51-be02-17d92d8f32bc',
 16: 'e9507868-8126-485b-a9f0-32c069e4a309',
 17: '64390817-b464-446c-a059-c03c9c46cf7a',
 18: 'fc23bc9e-79d5-4b84-91a2-760919f91769',
 19: '4850ff0a-ea89-4e6c-ab0a-7ea0b9acc2b2',
 20: 'de5ee9c8-d0e9-4dfe-be82-f4a4ade788ce',
 21: '84e42caf-5170-4774-9117-c640c769f0f0',
 22: '20d0c867-9ee2-

In [12]:
vector_store.get_by_ids(['49310c4a-5489-47c7-81f9-ff6a272490da'])

[]

## Step2 - Retrival

In [14]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [15]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002E7B7154830>, search_kwargs={'k': 4})

In [16]:
retriever.invoke('Why Math is beautiful')  # Because retriver is runnable function we can use invoke function

[Document(id='852bc9f5-e31b-4e59-97f6-821a7030c98c', metadata={}, page_content="Suppose you love math, and you had to choose just one proof to show someone to explain  why it is that math is beautiful, something that can be appreciated by anyone from a wide  range of backgrounds while still capturing the spirit of progress and cleverness in math. What would you choose? After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses,  published as a guest video over on MinutePhysics,  someone on Reddit asked about why the definition of an ellipse given"),
 Document(id='2397954f-2db1-4f34-b7da-7c3de0daa0a0', metadata={}, page_content='but more than that,  it reflects a common feature of math that sometimes there is no single  most fundamental way of defining something, that what matters more is  showing equivalences. And even more than that, the proof itself involves one key moment  of creative construction, adding the two spheres,  while most of it leaves room for 

## Step 3 - Augmentation

In [18]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [19]:
question = "Why math is beautiful ?"
retrieved_docs  = retriever.invoke(question)

In [20]:
retrieved_docs

[Document(id='852bc9f5-e31b-4e59-97f6-821a7030c98c', metadata={}, page_content="Suppose you love math, and you had to choose just one proof to show someone to explain  why it is that math is beautiful, something that can be appreciated by anyone from a wide  range of backgrounds while still capturing the spirit of progress and cleverness in math. What would you choose? After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses,  published as a guest video over on MinutePhysics,  someone on Reddit asked about why the definition of an ellipse given"),
 Document(id='a1492eab-a21b-4cc7-9218-4fda3cd5c4b7', metadata={}, page_content="definition of an ellipse as a stretched circle is the same as the  other two. More homework. So why do I think this proof is such a good representative for math itself? That if you had to show just one thing to explain to a non-math  enthusiast why you love the subject, why this would be a good candidate. The obvious reason is that it'

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs) # it Will merge all 4 document 
context_text

"Suppose you love math, and you had to choose just one proof to show someone to explain  why it is that math is beautiful, something that can be appreciated by anyone from a wide  range of backgrounds while still capturing the spirit of progress and cleverness in math. What would you choose? After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses,  published as a guest video over on MinutePhysics,  someone on Reddit asked about why the definition of an ellipse given\n\ndefinition of an ellipse as a stretched circle is the same as the  other two. More homework. So why do I think this proof is such a good representative for math itself? That if you had to show just one thing to explain to a non-math  enthusiast why you love the subject, why this would be a good candidate. The obvious reason is that it's substantive and beautiful without  requiring too much background, but more than that,  it reflects a common feature of math that sometimes there is no single

In [22]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

## Step 4 - Generation

In [24]:
llm = ChatOllama(model="qwen3:8b", temperature=0.2)

In [25]:
answer = llm.invoke(final_prompt)
print(answer.content)

Math is beautiful because it reveals profound equivalences between seemingly different concepts, showcasing the interconnectedness of ideas. The example of the ellipse—proving that a stretched circle (geometric definition) is equivalent to the focus-directrix definition—exemplifies this. The beauty lies in the **creative construction** (e.g., adding two spheres to visualize the proof) and the **systematic rigor** that bridges intuitive and abstract definitions. This reflects mathematics' essence: transforming complexity into clarity through ingenuity and structured reasoning, making it accessible yet deeply satisfying for all.


## Building a Chain

In [27]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [28]:
# Here we make function to join all 4 context because latter we have to convert it into Runnable so...
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [29]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),   # Take question -> Using retriver retive 4 document --> merge it
    'question': RunnablePassthrough()
})

In [30]:
parallel_chain.invoke('Why math is beautiful ?')

{'context': "Suppose you love math, and you had to choose just one proof to show someone to explain  why it is that math is beautiful, something that can be appreciated by anyone from a wide  range of backgrounds while still capturing the spirit of progress and cleverness in math. What would you choose? After I put out a video on Feynman's Lost Lecture about why planets orbit in ellipses,  published as a guest video over on MinutePhysics,  someone on Reddit asked about why the definition of an ellipse given\n\ndefinition of an ellipse as a stretched circle is the same as the  other two. More homework. So why do I think this proof is such a good representative for math itself? That if you had to show just one thing to explain to a non-math  enthusiast why you love the subject, why this would be a good candidate. The obvious reason is that it's substantive and beautiful without  requiring too much background, but more than that,  it reflects a common feature of math that sometimes there 

In [31]:
parser = StrOutputParser()

In [32]:
main_chain = parallel_chain | prompt | llm | parser

In [33]:
main_chain.invoke('Can you summarize the video')

'The video explains why the classic "two thumbtacks and string" definition of an ellipse is equivalent to the "slicing a cone" definition. It highlights a elegant 3D geometric proof by Dandelin (1822), where spheres tangent to a cone and a cutting plane reveal that an ellipse has the constant sum of distances property (focal definition). This connects the thumbtack construction to conic sections, showcasing mathematical ingenuity through minimal background.'